<a href="https://colab.research.google.com/github/MTalha-786/RAG_pro/blob/main/AI_Internship_2.0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Task 1: News Topic Classifier Using BERT**

---

**Objective:**

Fine-tune a transformer model (e.g., BERT) to classify news headlines into topic categories.
Dataset:
AG News Dataset (Available on Hugging Face Datasets)


**Instructions:**


● Tokenize and preprocess the dataset

● Fine-tune the bert-base-uncased model using Hugging Face Transformers

● Evaluate the model using accuracy and F1-score

● Deploy the model using Streamlit or Gradio for live interaction

---

---


In [ ]:
!pip install transformers datasets evaluate accelerate scikit-learn gradio torch

In [ ]:
import numpy as np
import pandas as pd
import torch
import evaluate
import gradio as gr
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# ==========================================
# 1. DATASET LOADING & SUBSET SELECTION
# ==========================================

# Download the AG News dataset from Hugging Face
datasets = load_dataset("ag_news")

# Shuffle and slice a smaller subset to optimize training time and compute resources
small_train_data = datasets["train"].shuffle(seed=42).select(range(5000))
small_test_data = datasets["test"].shuffle(seed=42).select(range(1000))

# ==========================================
# 2. TOKENIZATION & PREPROCESSING
# ==========================================

# Load the pre-trained BERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Define mapping function to truncate/pad headlines to a consistent length
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=64)

# Apply tokenization across the subsets in batches
tokenized_train = small_train_data.map(tokenize_function, batched=True)
tokenized_test = small_test_data.map(tokenize_function, batched=True)

# ==========================================
# 3. MODEL CONFIGURATION & TRAINING ARGS
# ==========================================

# Initialize BERT sequence classifier with 4 target categories
model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

# Configure model training hyperparameters
training_args = TrainingArguments(
    output_dir="./results",          # Directory where execution checkpoints are saved
    eval_strategy="epoch",           # Evaluate performance metrics at the end of each epoch
    learning_rate=2e-5,              # Initial learning rate optimized for BERT fine-tuning
    per_device_train_batch_size=16,  # Batch size for training execution
    per_device_eval_batch_size=16,   # Batch size for evaluation validation
    num_train_epochs=2,              # Total number of training passes over the dataset
    weight_decay=0.01,               # Strength of weight decay regularization to avoid overfitting
)

# ==========================================
# 4. EVALUATION METRICS DEFINITION
# ==========================================

# Load standard accuracy and F1 metrics tracking frameworks
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    # Calculate performance metrics
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="weighted")["f1"]

    return {"accuracy": acc, "f1": f1}

# ==========================================
# 5. MODEL TRAINING & SAVING FLOW
# ==========================================

# Package the configured architecture components inside the Trainer framework
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

# Execute the fine-tuning loop
print("Starting training process...")
trainer.train()

# Persist the fine-tuned model weights and matching tokenization configurations locally
print("Saving fine-tuned model artifacts...")
model.save_pretrained("./my_bert_model")
tokenizer.save_pretrained("./my_bert_model")
print("Model artifacts successfully saved!")

# ==========================================
# 6. APP INFERENCE LOGIC (GRADIO WEB DEPLOYMENT)
# ==========================================

# Load the locally stored fine-tuned model and tokenizer artifacts for inference
model_path = "./my_bert_model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

# Define explicit descriptive targets based on the AG News class index map
labels = ["World", "Sports", "Business", "Sci/Tech"]

def classify_news(headline):
    # Convert string text sequence to tensor inputs wrapped for PyTorch execution
    inputs = tokenizer(headline, return_tensors="pt", truncation=True, padding=True, max_length=64)

    # Evaluate input tokens without building gradients to ensure faster pipeline calculation
    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        # Convert logits array into soft probability scores ranging from 0 to 1
        probabilities = torch.nn.functional.softmax(logits, dim=-1)[0]

    # Return dictionary structured with readable tags matched with confidence distribution scores
    return {labels[i]: float(probabilities[i]) for i in range(4)}

# Configure the front-end layout components for the interactive interface
demo = gr.Interface(
    fn=classify_news,
    inputs=gr.Textbox(lines=2, placeholder="Type an English news headline here...", label="News Headline"),
    outputs=gr.Label(num_top_classes=4, label="Predicted Category"),
    title="📰 News Topic Classifier (BERT)",
    description="Fine-tuned BERT model classifying news headlines into World, Sports, Business, or Sci/Tech classes."
)

# Main runtime block handler to initialize web server engine
if __name__ == "__main__":
    demo.launch()

**Task 2: End-to-End ML Pipeline with Scikit-learn Pipeline API**

---

**Objective:**

Build a reusable and production-ready machine learning pipeline for predicting customer
churn.
Dataset:
Telco Churn Dataset

**Instructions:**

● Implement data preprocessing steps (e.g., scaling, encoding) using Pipeline

● Train models like Logistic Regression and Random Forest

● Use GridSearchCV for hyperparameter tuning

● Export the complete pipeline using joblib


---


---



In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score, classification_report

import joblib

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
df = pd.read_csv("/content/WA_Fn-UseC_-Telco-Customer-Churn.csv")

df.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [ ]:
# Drop customerID (not useful)
df.drop("customerID", axis=1, inplace=True)

# Convert TotalCharges to numeric (common issue in this dataset)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Target column conversion
df["Churn"] = df["Churn"].map({"Yes": 1, "No": 0})

In [ ]:
X = df.drop("Churn", axis=1)
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object"]).columns

In [ ]:
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

In [ ]:
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [ ]:
log_reg_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier())
])

In [ ]:
log_reg_pipeline.fit(X_train, y_train)
rf_pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('encoder',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                ('model', RandomForestClassifier())])

In [ ]:
y_pred_lr = log_reg_pipeline.predict(X_test)

print("Logistic Regression Accuracy:", accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

Logistic Regression Accuracy: 0.8055358410220014
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.66      0.56      0.60       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [ ]:
y_pred_rf = rf_pipeline.predict(X_test)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Random Forest Accuracy: 0.7799858055358411
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1035
           1       0.61      0.48      0.54       374

    accuracy                           0.78      1409
   macro avg       0.72      0.69      0.70      1409
weighted avg       0.77      0.78      0.77      1409



In [ ]:
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 10, 20],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=3,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

GridSearchCV(cv=3,
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(transformers=[('num',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='median')),
                                                                                         ('scaler',
                                                                                          StandardScaler())]),
                                                                         Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object')),
                                                                        ('cat',
                                                                         Pipeline(steps=[('imputer',
                                                                                          SimpleImputer(strategy='most_frequent')),
                                                                                         ('e...
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object'))])),
                                       ('model', RandomForestClassifier())]),
             n_jobs=-1,
             param_grid={'model__max_depth': [None, 10, 20],
                         'model__min_samples_split': [2, 5],
                         'model__n_estimators': [100, 200]},
             scoring='accuracy')

In [ ]:
print("Best Parameters:", grid_search.best_params_)
print("Best Score:", grid_search.best_score_)

Best Parameters: {'model__max_depth': 10, 'model__min_samples_split': 5, 'model__n_estimators': 100}
Best Score: 0.8044018459353923


In [ ]:
best_model = grid_search.best_estimator_

joblib.dump(best_model, "churn_pipeline.pkl")

['churn_pipeline.pkl']